# 03 · Versión B: afinar el LLM de visión con LoRA

## Qué vas a aprender

- Qué es **LoRA** y por qué permite afinar un modelo de 256 millones de parámetros en un portátil sin GPU.
- Qué se **congela** y qué se **entrena** (1 843 200 parámetros, el 0,71 %).
- A leer una **curva de pérdida** y a desconfiar de ella: puede bajar y el modelo seguir sin hacer la tarea.
- A registrar el **adaptador** (7 MB) en MLflow, aparte del modelo base (494 MB).

## Dónde estamos

En el notebook 02 el modelo sin entrenar (**A**) describía la foto o copiaba los rangos del prompt. Ahora le enseñamos con **210 ejemplos** (foto editada + los cinco ajustes reales) a contestar con un JSON. **Fine-tuning** es seguir entrenando un modelo ya entrenado con nuestros datos.

> **Aviso de tiempo.** Este notebook tarda unos **24 minutos en CPU** con las 300 fotos y 3 épocas (medido el 20/11/2026 en un portátil de 12 núcleos, sin GPU). Si lo lees sin ejecutar, no te pierdes nada: debajo de cada celda de código hay un recuadro con la **salida del estudio técnico del 20/11/2026**. Si lo ejecutas, verás tus propias cifras, parecidas pero no idénticas.

Entrada: `parejas/` de la carpeta de trabajo. Salida: `adaptador/`, `resultados/pred_B.jsonl`, `resultados/metricas_B.json`, `mlflow.db` (con un run `B_lora`).

In [ ]:
import os
N_EJEMPLOS = int(os.environ.get('ESTILO_N_EJEMPLOS', '60'))
EPOCAS = int(os.environ.get('ESTILO_EPOCAS', '2'))
RESOLUCION = int(os.environ.get('ESTILO_RESOLUCION', '224'))
MODELO = os.environ.get('ESTILO_MODELO', 'HuggingFaceTB/SmolVLM-256M-Instruct')
CARPETA_TRABAJO = os.environ.get('ESTILO_TRABAJO', '_trabajo')
TAM_LOTE = 4
TASA_APRENDIZAJE = 5e-4
RANGO_LORA = 16

## 1. Qué es LoRA

Un modelo de lenguaje está hecho de muchas **matrices de pesos** grandes. Entrenarlas todas es lento y obliga a guardar una copia entera del modelo por cada tarea. **LoRA** (*Low-Rank Adaptation*, «adaptación de rango bajo») hace otra cosa:

- La matriz original `W` **no se toca** (se queda congelada).
- A su lado se añaden dos matrices **pequeñas**, `A` y `B`, que **sí** se entrenan.
- La capa calcula `W·x + B·A·x`: lo que ya sabía, más una **corrección** aprendida.

```text
                    entrada x
                   /         \
                  v           v
        +--------------+   +--------------+
        |      W       |   |  A  (r x 576)|   <- pequeña, SE ENTRENA
        |  576 x 576   |   +--------------+
        |  CONGELADA   |          |
        +--------------+          v
                  |        +--------------+
                  |        |  B  (576 x r)|   <- pequeña, SE ENTRENA
                  |        +--------------+
                  v                v
                   \              /
                    +---- suma ---+
                          |
                    salida = W·x + B·A·x          (r = 16 en este notebook)
```

Una analogía: `W` es un libro impreso que no se puede borrar; `A` y `B` son las notas a lápiz de los márgenes. Si cambias de tarea, tiras las notas y escribes otras; el libro es el mismo.

Lo que guardamos al final es **solo el adaptador** (las notas), no el libro.

## 2. Qué se congela y qué se entrena

El LLM de visión SmolVLM-256M tiene tres piezas (cifras del estudio técnico):

| Pieza | Tamaño | Aquí |
|---|---|---|
| Codificador de imagen (SigLIP) | 86 M parámetros | **congelado** (ya sabe ver) |
| Conector imagen → texto | 7 M parámetros | **congelado** |
| Modelo de lenguaje (SmolLM2) | 135 M parámetros | **congelado**, con LoRA en la atención |

Lo que hay que enseñar es a **escribir** cinco números, no a ver. Por eso ponemos LoRA solo en las cuatro proyecciones de la atención del modelo de lenguaje: `q_proj`, `k_proj`, `v_proj` y `o_proj`.

**Qué debes ver** al ejecutar la celda de abajo: una línea `trainable params: 1,843,200 || all params: 258,328,128 || trainable%: 0.7135`.

Se puede comprobar a mano. El modelo de lenguaje tiene 30 capas, tamaño oculto 576, 9 cabezas de atención y 3 cabezas de claves/valores de 64 dimensiones cada una (de su `config.json`). Para cada matriz, LoRA añade `r × (entradas + salidas)` parámetros, con `r` = 16:

- `q_proj` y `o_proj` (576 → 576): 16 × (576 + 576) = 18 432 cada una.
- `k_proj` y `v_proj` (576 → 192): 16 × (576 + 192) = 12 288 cada una.
- Por capa: 18 432 + 12 288 + 12 288 + 18 432 = 61 440. En 30 capas: **1 843 200**. Cuadra con lo que imprime PEFT.

In [ ]:
import json, random
import numpy as np
import matplotlib.pyplot as plt
import estilo as ce
from estilo.vlm import preparar_ejemplo, lote

plt.rcParams['figure.dpi'] = 55
trabajo = ce.carpeta_trabajo(CARPETA_TRABAJO)   # antes de importar transformers: fija la caché de Hugging Face
hilos = ce.limitar_hilos()
import torch, transformers, peft
from peft import LoraConfig, get_peft_model
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| hilos:', hilos)
torch.manual_seed(0); random.seed(0)

proc, vlm = ce.cargar_vlm(MODELO)
for p in vlm.parameters():          # 1) congelamos TODO el modelo...
    p.requires_grad = False
config = LoraConfig(r=RANGO_LORA, lora_alpha=2 * RANGO_LORA, lora_dropout=0.05,
                    target_modules=r'.*text_model.*\.(q_proj|k_proj|v_proj|o_proj)')
modelo = get_peft_model(vlm, config)  # 2) ...y añadimos las matrices LoRA, que sí se entrenan
modelo.print_trainable_parameters()

> **Salida del estudio técnico del 20/11/2026** (portátil sin GPU, 12 hilos):
> ```text
> torch 2.15.0+cpu | transformers 5.23.0 | peft 0.22.0 | hilos: 12
> trainable params: 1,843,200 || all params: 258,328,128 || trainable%: 0.7135
> ```

**La configuración, en una tabla:**

| Ajuste | Valor | Para qué sirve |
|---|---|---|
| `r` (rango) | 16 | tamaño de las matrices `A` y `B`: más rango, más capacidad y más parámetros |
| `lora_alpha` | 32 (= 2·r) | cuánto pesa la corrección frente al modelo original |
| `lora_dropout` | 0,05 | apaga al azar el 5 % durante el entrenamiento para no memorizar |
| Módulos | `q_proj`, `k_proj`, `v_proj`, `o_proj` | las cuatro proyecciones de la atención del modelo de lenguaje |
| Tasa de aprendizaje | 5e-4 | tamaño del paso del optimizador (AdamW) |
| Lote | 4 ejemplos | ejemplos por paso |

**Si falla:** un `OSError` o un error de red al cargar el modelo es el proxy o la caché de Hugging Face (mira el notebook 00). Un `MemoryError` o un proceso «Killed» es falta de memoria: el entrenamiento llegó a 2,7 GB de pico en el estudio técnico.

## 3. Preparar los ejemplos

Cada ejemplo es: la imagen + la orden corta `Color edits as JSON.` + la respuesta correcta (el JSON). La pérdida solo se calcula sobre la **respuesta**: no queremos que aprenda a repetir el prompt.

**Truco de CPU.** Como el codificador de imagen está congelado, las «palabras de imagen» de cada foto son siempre las mismas. Las calculamos **una vez** aquí y las reutilizamos en todas las épocas. Es la mitad del tiempo de cada paso.

In [ ]:
train, val, test = (ce.leer_parejas(trabajo, p) for p in ('train', 'val', 'test'))
with ce.Cronometro() as c_prep:
    ej_train = [preparar_ejemplo(proc, vlm, ce.abrir(trabajo, f['editada']), f['ajustes']) for f in train]
    ej_val = [preparar_ejemplo(proc, vlm, ce.abrir(trabajo, f['editada']), f['ajustes']) for f in val]
n_resp = int((ej_train[0]['etiquetas'] != -100).sum())
print(f'{len(ej_train)} de entrenamiento y {len(ej_val)} de validación preparados en {c_prep.s:.0f} s')
print(f"Longitud de un ejemplo: {len(ej_train[0]['ids'])} tokens, de ellos {n_resp} de respuesta")
print('Texto de un ejemplo (sin las 64 palabras de imagen):')
print(proc.tokenizer.decode([t for t in ej_train[0]['ids'] if t != vlm.config.image_token_id]))

> **Salida del estudio técnico del 20/11/2026** (210 de entrenamiento, 30 de validación):
> ```text
> 210 de entrenamiento y 30 de validación preparados en 530 s
> Longitud de un ejemplo: 135 tokens, de ellos 55 de respuesta
> Texto de un ejemplo (sin las 64 palabras de imagen):
> <|im_start|>User:<fake_token_around_image><global-img><fake_token_around_image>Color edits as JSON.<end_of_utterance>
> Assistant: {"exposicion": 0.13, "temperatura": -0.67, "tinte": -0.88, "contraste": 0.92, "saturacion": 1.02}<end_of_utterance>
> ```

Preparar los ejemplos costó 530 s (unos 2,2 s por foto): es el coste del codificador de imagen, que ya no se paga en cada época.

## 4. Entrenar

Un bucle de PyTorch normal: lote → pérdida → gradientes → paso del optimizador. La **pérdida** es la entropía cruzada de predecir cada token de la respuesta: cuanto más baja, más probable le parece al modelo la respuesta correcta. Al final de cada época la medimos también en **validación** (fotos que el modelo no usa para aprender).

In [ ]:
pad = proc.tokenizer.pad_token_id
optimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=TASA_APRENDIZAJE)

def perdida_val():
    modelo.eval()
    with torch.no_grad():
        return float(np.mean([modelo(**lote(ej_val[i:i + 8], pad)).loss.item() for i in range(0, len(ej_val), 8)]))

historia = {'paso': [], 'perdida': [], 'epoca': [0], 'perdida_val': [perdida_val()]}
print(f"Época 0: pérdida de validación {historia['perdida_val'][0]:.3f}")
with ce.Cronometro() as c_train:
    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        random.shuffle(ej_train)
        for i in range(0, len(ej_train), TAM_LOTE):
            perdida = modelo(**lote(ej_train[i:i + TAM_LOTE], pad)).loss
            perdida.backward(); optimizador.step(); optimizador.zero_grad()
            historia['paso'].append(len(historia['paso']) + 1); historia['perdida'].append(perdida.item())
        historia['epoca'].append(epoca); historia['perdida_val'].append(perdida_val())
        print(f"Época {epoca}: pérdida de entrenamiento (último lote) {perdida.item():.3f}, "
              f"validación {historia['perdida_val'][-1]:.3f}")
pasos = len(historia['paso'])
print(f'Entrenamiento: {c_train.s:.0f} s, {pasos} pasos, {c_train.s / pasos:.2f} s por paso de {TAM_LOTE} ejemplos')

> **Salida del estudio técnico del 20/11/2026** (3 épocas):
> ```text
> Época 0: pérdida de validación 2.758
> Época 1: pérdida de entrenamiento (último lote) 0.465, validación 0.479
> Época 2: pérdida de entrenamiento (último lote) 0.474, validación 0.476
> Época 3: pérdida de entrenamiento (último lote) 0.484, validación 0.475
> Entrenamiento: 593 s, 159 pasos, 3.73 s por paso de 4 ejemplos
> ```

Del orden de **10 minutos** de entrenamiento en CPU para 159 pasos.

In [ ]:
pasos_por_epoca = pasos / EPOCAS
plt.figure(figsize=(7, 3.5))
plt.plot(historia['paso'], historia['perdida'], alpha=0.6, label='entrenamiento (cada lote)')
plt.plot([e * pasos_por_epoca for e in historia['epoca']], historia['perdida_val'], 'o-', label='validación (fin de época)')
plt.xlabel('paso'); plt.ylabel('pérdida'); plt.legend(); plt.title('Curva de pérdida (LoRA)')
plt.tight_layout()
(trabajo / 'resultados').mkdir(exist_ok=True)
ce.guardar_figura(trabajo / 'resultados' / 'curva_perdida.png')
plt.show()

**La curva del estudio técnico** (3 épocas, salida del 20/11/2026):

![Curva de pérdida de la LoRA: baja de 2,76 a 0,48 en la primera época y se queda plana](img/03_curva_perdida.png)

**Cómo se lee.**

- La pérdida cae de **2,758 a 0,479** en la primera época (53 pasos) y **ya no baja**: 0,476 y 0,475 en las dos siguientes.
- La de validación va pegada a la de entrenamiento: no hay sobreajuste.
- Una caída tan rápida y tan plana es típica de aprender el **formato**: tras 53 pasos el modelo ya sabe que debe escribir `{"exposicion": ..., ...}`. Es una lectura nuestra de la curva; lo que importa es lo que mide el test de abajo.

Parece un éxito: la pérdida se dividió casi por seis (de 2,758 a 0,479). **Todavía no sabemos si hace la tarea.**

## 5. Guardar el adaptador y medir en test

Guardamos **solo** el adaptador. Después hacemos lo que haría una aplicación real: a cada foto de test (60 fotos que el modelo no ha visto) le damos **la imagen** y la orden corta, y leemos el JSON que escribe.

Medimos con el **MAE normalizado** (error medio absoluto dividido por el rango de cada ajuste; 0 = perfecto). Y nos comparamos con una línea base tonta: **contestar siempre la media** de los ajustes de train. Un modelo que no mejora a la «media» no ha aprendido nada de la foto.

In [ ]:
carpeta_adaptador = trabajo / 'adaptador'
modelo.save_pretrained(carpeta_adaptador)
tam_adaptador = ce.tamano_mb(carpeta_adaptador / 'adapter_model.safetensors')
from huggingface_hub import snapshot_download
tam_base = ce.tamano_mb(snapshot_download(MODELO, allow_patterns=['*.safetensors']))
print(f'Adaptador: {tam_adaptador} MB  |  pesos del modelo base (safetensors): {tam_base} MB')

modelo.eval()
resultados, tiempos = [], []
for f in test:
    with ce.Cronometro() as c:
        respuesta = ce.preguntar(proc, modelo, ce.abrir(trabajo, f['editada']), prompt=ce.PROMPT_CORTO)
    tiempos.append(c.s)
    pred, completo = ce.parsear(respuesta)
    resultados.append({'id': f['id'], 'respuesta': respuesta, 'pred': pred, 'completo': completo})
(trabajo / 'resultados' / 'pred_B.jsonl').write_text(
    '\n'.join(json.dumps(r, ensure_ascii=False) for r in resultados), encoding='utf-8')

> **Salida del estudio técnico del 20/11/2026:**
> ```text
> Adaptador: 7.06 MB  |  pesos del modelo base (safetensors): 493.9 MB
> ```

El adaptador pesa **7,06 MB** frente a los **493,9 MB** del modelo base: unas 70 veces menos. Es lo único que hay que guardar y mover (por ejemplo, como artefacto de GitHub Actions) para tener el modelo afinado.

In [ ]:
reales = [f['ajustes'] for f in test]
m = ce.error_por_ajuste([r['pred'] for r in resultados], reales)
media = ce.error_por_ajuste([ce.ajustes_medios(train)] * len(test), reales)
m.update(formato_correcto=float(np.mean([r['completo'] for r in resultados])),
         segundos_por_foto=float(np.mean(tiempos)), segundos_entrenamiento=c_train.s,
         segundos_preparacion=c_prep.s, adaptador_mb=tam_adaptador, base_mb=tam_base,
         memoria_pico_mb=ce.memoria_mb())
print(f"Formato correcto: {m['formato_correcto']:.0%}  |  {m['segundos_por_foto']:.2f} s por foto")
print(f"{'ajuste':12s} {'MAE':>6s} {'MAE norm.':>10s} {'«media» norm.':>14s}")
for k in ce.NOMBRES:
    print(f"{k:12s} {m['mae_' + k]:6.3f} {m['maen_' + k]:10.3f} {media['maen_' + k]:14.3f}")
print(f"{'MEDIA':12s} {'':6s} {m['mae_normalizado_medio']:10.3f} {media['mae_normalizado_medio']:14.3f}")
print('Memoria pico del proceso (MB):', m['memoria_pico_mb'])
(trabajo / 'resultados' / 'metricas_B.json').write_text(json.dumps(m, indent=1), encoding='utf-8')

**Resultados en test del estudio técnico del 20/11/2026** (60 fotos; 3 épocas; `neutro` = contestar «no se tocó nada»; la columna «media» del estudio coincide con `neutro` en la media, 0,259):

| Ajuste | MAE normalizado de B | Contestar siempre lo mismo (`neutro`) |
|---|---|---|
| exposición | 0,299 | 0,262 |
| temperatura | 0,342 | 0,250 |
| tinte | 0,285 | 0,249 |
| contraste | 0,317 | 0,242 |
| saturación | 0,308 | 0,290 |
| **Media** | **0,310** | **0,259** |

- **Formato correcto: 100 %.** Las 60 respuestas son un JSON con los cinco ajustes.
- **Error: 0,310**, peor que contestar siempre la media (**0,259**) y mejor que A (0,433, el modelo sin entrenar).
- **Inferencia: 4,91 s por foto.** Memoria pico del proceso: 2 747 MB.
- **Tiempo total:** 530 s de preparación + 593 s de entrenamiento = 1 123 s (unos 19 minutos) y el notebook entero, 24 minutos.

El modelo hace el formato **perfecto** y falla **en los cinco ajustes**. Veamos por qué.

## 6. Diagnóstico: aprende el formato, no la tarea

Si el modelo mirase cada foto, sus cinco números cambiarían de una foto a otra, como los reales. Comprobémoslo con tres medidas:

1. cuántas respuestas **distintas** escribe en 60 fotos;
2. cuánto **varían** sus números (desviación típica) frente a cuánto varían los reales;
3. si sus números **siguen** a los reales (correlación: 1 = los sigue, 0 = no tiene relación).

In [ ]:
from collections import Counter
cuenta = Counter(r['respuesta'].strip() for r in resultados)
print(f'{len(resultados)} fotos de test y solo {len(cuenta)} respuestas distintas. Las dos más repetidas:')
for texto, n in cuenta.most_common(2):
    print(f'  {n:2d} veces: {texto}')
print()
print(f"{'ajuste':12s} {'desv. real':>11s} {'desv. modelo':>13s} {'correlación':>12s}")
for k in ce.NOMBRES:
    real, modelo_k = [f['ajustes'][k] for f in test], [r['pred'][k] for r in resultados]
    corr = float(np.corrcoef(real, modelo_k)[0, 1]) if np.std(modelo_k) > 0 else float('nan')
    print(f'{k:12s} {np.std(real):11.3f} {np.std(modelo_k):13.3f} {corr:12.2f}')

> **Salida calculada sobre `pred_B.jsonl` de la ejecución del estudio técnico del 20/11/2026:**
> ```text
> 60 fotos de test y solo 15 respuestas distintas. Las dos más repetidas:
>   22 veces: {"exposicion": 0.32, "temperatura": 0.62, "tinte": 0.62, "contraste": 0.82, "saturacion": 0.68}
>   19 veces: {"exposicion": 0.32, "temperatura": 0.62, "tinte": 0.62, "contraste": 0.82, "saturacion": 0.6}
> 
> ajuste        desv. real  desv. modelo  correlación
> exposicion         0.432         0.065         0.06
> temperatura        0.579         0.193         0.03
> tinte              0.564         0.100         0.15
> contraste          0.250         0.013        -0.04
> saturacion         0.516         0.039         0.19
> ```

**Qué significa:**

- **Dos respuestas casi iguales cubren 41 de las 60 fotos** (22 y 19), y solo hay 15 distintas. El modelo escribe «el JSON más probable», no el de cada foto.
- **Casi no varía:** en contraste, sus números se mueven 0,013 cuando los reales se mueven 0,250.
- **No sigue a los reales:** las correlaciones están entre −0,04 y 0,19, es decir, **cerca de cero**. Con solo 60 fotos, valores tan pequeños no se distinguen del azar.

Esto es **aprender el formato pero no la tarea**. La pérdida bajó porque predecir las llaves, las comillas y los números típicos es fácil; predecir el número concreto de cada foto, no.

### Respuestas reales: tres fotos, casi la misma respuesta

Tres fotos de test con distinta edición (salida del estudio técnico del 20/11/2026). Fíjate en cada columna:

| | exposición | temperatura | tinte | contraste | saturación |
|---|---|---|---|---|---|
| Foto 1, real | 0,45 | 0,5 | 0,03 | 1,35 | 1,69 |
| Foto 1, B escribe | 0,22 | 0,62 | 0,62 | 0,82 | 0,68 |
| Foto 2, real | −0,58 | −0,08 | 0,54 | 0,67 | 1,73 |
| Foto 2, B escribe | 0,32 | 0,62 | 0,62 | 0,82 | 0,6 |
| Foto 3, real | −0,18 | −0,93 | 0,44 | 1,37 | 0,6 |
| Foto 3, B escribe | 0,22 | 0,62 | 0,62 | 0,82 | 0,68 |

- La foto 1 y la foto 3 reciben **exactamente la misma respuesta**, aunque una se calentó (+0,5) y la otra se enfrió (−0,93).
- En la foto 2 se bajó la exposición (−0,58) y el modelo escribe **+0,32**: sentido contrario.
- La saturación real es 1,69 y 1,73 (más color) y el modelo escribe 0,68 y 0,6 (menos color).

## 7. Último intento: darle los números masticados (variante 2c)

Quizá el problema es que el modelo no «ve» bien los colores a través de sus 64 palabras de imagen (hipótesis nuestra, sin verificar). Probamos lo contrario: **no le damos la imagen**, sino sus **estadísticas de color escritas como texto en el prompt**, y lo afinamos con LoRA igual que antes. La celda de abajo muestra ese prompt para una foto de test (no necesita el modelo).

El resultado de la variante **2c** (ronda 2 del estudio técnico, 20/11/2026) está citado del informe del estudio; aquí no se reentrena:

In [ ]:
foto = ce.abrir(trabajo, test[0]['editada'])
print(ce.prompt_con_estadisticas(foto))

**Cómo se hizo 2c** (estudio técnico, ronda 2): mismo modelo de lenguaje, **sin imagen**, con el prompt de arriba; la misma LoRA (r = 16, q/k/v/o, tasa 5e-4, lote 4) y **2 épocas** con las mismas 210 fotos de train; evaluada en las mismas 60 de test. Entrenamiento: 494 s. Inferencia: 3,1 s por foto (no incluye imagen).

| Versión | MAE normalizado (60 fotos de test) | Formato correcto |
|---|---|---|
| Contestar siempre la media | 0,259 | – |
| **B** (imagen → JSON, 3 épocas) | **0,310** | 100 % |
| **2c** (estadísticas en el prompt → JSON, 2 épocas) | **0,244** | 100 % |
| 2d (imagen + estadísticas, 1 época por tiempo) | 0,262 | 100 % |
| Una regresión sobre esas mismas estadísticas (`C_color`, notebook 04) | **0,151** | – |

**Es la única versión del LLM que mejora a la «media», y por poco**: 0,015 de MAE normalizado, con 60 fotos de test y una sola semilla. La mejora está sobre todo en contraste (0,153 frente a 0,240 de la media); en saturación es peor que la media (0,345). Escribe 32 respuestas distintas en 60 fotos (B escribe 15): ahora al menos depende de la entrada.

Y **tampoco basta**: queda a **0,09** de `C_color`, que hace lo mismo con una regresión lineal que entrena en segundos. Con un LLM de 135 M y 2 épocas, leer los números no supera a la regresión que recibe esos mismos números.

Otras pruebas del estudio técnico, para descartar que el fallo fuera de configuración: LoRA en **todas** las capas lineales, tasa 1e-3 y 4 épocas dio 0,303 frente a 0,285 de la media en 20 fotos de validación; pedir tres clases («bajo/normal/alto») en lugar de números dio una exactitud del 32 %, al nivel del azar (33 %). No se probaron otros tamaños de modelo (SmolVLM-500M), más épocas ni entrenar el conector: **sin verificar**.

## 8. Registrar en MLflow

Un *run* con los hiperparámetros, la pérdida paso a paso, las métricas de test y el **adaptador como artefacto**. Así, en la S4 podemos comparar runs y recuperar el adaptador de un run concreto. La base es **`mlflow.db` en la carpeta de trabajo** (SQLite, sin servidor), no la del proyecto del curso.

In [ ]:
mlflow = ce.mlflow_local(trabajo)
with mlflow.start_run(run_name='B_lora') as run:
    mlflow.log_params({'version': 'B', 'modelo': MODELO, 'rango_lora': RANGO_LORA, 'epocas': EPOCAS,
                       'tam_lote': TAM_LOTE, 'tasa_aprendizaje': TASA_APRENDIZAJE, 'n_train': len(train),
                       'n_val': len(val), 'n_test': len(test), 'hilos': hilos, 'prompt': ce.PROMPT_CORTO})
    for paso, valor in zip(historia['paso'], historia['perdida']):
        mlflow.log_metric('perdida_train', valor, step=paso)
    for epoca, valor in zip(historia['epoca'], historia['perdida_val']):
        mlflow.log_metric('perdida_val', valor, step=epoca)
    mlflow.log_metrics(m)
    mlflow.log_artifacts(str(carpeta_adaptador), artifact_path='adaptador')
print('Run registrado:', run.info.run_id, '| experimento pico-llm-estilo')
print('Artefactos:', [a.path for a in mlflow.MlflowClient().list_artifacts(run.info.run_id, 'adaptador')])
print('Base de datos:', ce.mostrar_ruta(trabajo / 'mlflow.db', trabajo))

> **Salida del estudio técnico del 20/11/2026** (el identificador del run cambia en cada ejecución):
> ```text
> Run registrado: 625932b6d49f4480bdac93f73e9d36b4 | experimento pico-llm-estilo
> Artefactos: ['adaptador/README.md', 'adaptador/adapter_config.json', 'adaptador/adapter_model.safetensors']
> ```

**Qué debes ver** en la interfaz de MLflow (`mlflow ui --backend-store-uri sqlite:///<carpeta de trabajo>/mlflow.db`): un run `B_lora` con la curva `perdida_train`, la métrica `mae_normalizado_medio` y los tres archivos del adaptador. En el notebook 04 registraremos junto a él los demás runs y, de la versión ganadora, un modelo en el *registry*.

**Si falla:** `sqlite3.OperationalError: database is locked` es que otra ventana de MLflow tiene la base abierta; ciérrala.

## Qué te llevas

- **LoRA** congela el modelo y entrena matrices pequeñas: aquí 1 843 200 parámetros (0,71 %), un adaptador de **7 MB** en vez de copiar 494 MB. Cabe en un portátil: unos 19 minutos de preparación y entrenamiento con 210 fotos.
- **Una pérdida que baja y un formato perfecto no prueban que el modelo haya aprendido la tarea.** B escribe JSON válido el 100 % de las veces y se equivoca más (0,310) que contestar siempre la media (0,259): casi la misma respuesta para fotos muy distintas.
- **Dale los números al modelo** (variante 2c) lo mejora a 0,244, pero sigue lejos de una regresión sobre esas mismas estadísticas.
- **Registrar** el adaptador en MLflow deja el experimento reproducible aunque el resultado sea malo. Un resultado negativo bien medido también es un resultado.

## Preguntas para pensar

1. La pérdida de validación bajó de 2,758 a 0,475 y se quedó plana, y el modelo no mejora a la media. ¿Qué métrica añadirías **antes** de entrenar para detectarlo sin esperar 24 minutos?
2. Si tuvieras 100 veces más fotos y una GPU, ¿esperarías que B superase a la media? ¿Qué probarías primero y cómo sabrías que ha funcionado? (En el estudio técnico esto **no se probó**.)

**Siguiente:** el notebook 04 pone a B junto a una regresión sobre estadísticas de color, una red pequeña y un método clásico, y transfiere el estilo de verdad.